# exp01 - PharmaSales daily: perbandingan di bawah protokol tunggal

Notebook ini **menggantikan** perbandingan lintas-notebook pada revisi sebelumnya
(`our_study_pharma_daily.ipynb` vs `rathipriya_pharma_daily.ipynb`), yang oleh kedua
reviewer IJIES dinilai tidak sah karena setiap model dijalankan dengan preprocessing,
pembagian data, dan protokol tuning yang berbeda.

## Kontrak eksperimen

Semua angka di notebook ini dihasilkan di bawah satu kontrak yang dikodekan di
`src/experiments/protocol.py` (satu-satunya sumber kebenaran; notebook lain memakai
modul yang sama sehingga protokolnya tidak mungkin menyimpang):

| Aspek | Ketentuan |
|---|---|
| Seed | `SEED = 42`, dipasang ke `random`, NumPy, `PYTHONHASHSEED`; setiap estimator stokastik menerima `random_state=SEED` melalui satu konstruktor `make_xgb()` |
| Split | Kronologis 70 / 15 / 15 (train / validation / test), tanpa pengacakan, **identik untuk semua model dan semua set fitur** |
| Tuning | Hyperparameter dipilih **hanya** dari RMSE validation. Test split disentuh **satu kali** oleh model final |
| Refit | Model final di-refit pada train+val dengan konfigurasi terbaik dari validation |
| Fitur | Set fitur adalah **faktor eksperimen**: `A_lag1` (protokol referensi) dan `B_rich` (lag terpilih + rolling mean) - setiap model dijalankan pada keduanya |
| Seleksi lag | argmax PACF dihitung **hanya pada blok training**; aturan alternatif diuji sebagai ablasi tersendiri |
| Penskalaan | Scaler di-fit ulang pada blok training aktif saja, tidak pernah pada test |

## Perbedaan terhadap pipeline lama (dan alasannya)

1. **Baseline dan model usulan memakai fitur serta split yang sama.** Sebelumnya baris
   "Reference" pada Tabel 2 hanya diberi `lag_1` dengan split 70/15/15, sedangkan model
   usulan memakai 2-16 fitur dengan split 60/20/20. Selisih yang dilaporkan karena itu
   mencampur efek model dengan efek preprocessing.
2. **Tidak ada grid search pada test set.** Cabang "Our Preprocessing" pada notebook
   baseline memilih hyperparameter dengan menilai test split (80/20 tanpa validation).
3. **PACF dihitung pada blok training saja.** Notebook lama menghitung ACF/PACF pada
   seluruh deret termasuk test - kebocoran halus pada tahap desain fitur.
4. **Grid bandwidth kernel diperlebar.** Pada grid lama (maksimum sigma = 5) optimum
   validation jatuh persis di batas atas untuk sebagian besar kategori, artinya grid
   memotong ruang pencarian dan secara sistematis merugikan GRNN/P_NN.
5. **Baseline naif ditambahkan.** Klaim keunggulan tanpa pembanding naif tidak dapat
   dinilai; Naive dan Seasonal Naive (s=7) kini dilaporkan di setiap tabel.
6. **Uji Diebold-Mariano dilaporkan** agar selisih RMSE dapat dinilai signifikansinya,
   bukan sekadar dibandingkan angkanya.

In [ ]:
import sys, os, warnings
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.experiments import protocol as P

P.set_global_seed()

GRANULARITY      = "daily"
DATA_PATH        = "../data/raw/pharma-sales/salesdaily.csv"
SEASONAL_PERIOD  = 7
EXPERIMENT       = "exp01_pharma_daily"
CATEGORIES       = ["M01AB", "M01AE", "N02BA", "N02BE", "N05B", "N05C", "R03", "R06"]

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

data = pd.read_csv(DATA_PATH)
print("Lingkungan:", P.environment_stamp())
print("Baris data mentah:", len(data))
data.head()

## 1. Stabilitas aturan pemilihan lag

Nilai *k* (jumlah lag) menentukan seluruh set fitur `B_rich`. Notebook lama memakai
**dua aturan berbeda** untuk dua pipeline yang seharusnya dibandingkan: `our_study`
memakai argmax PACF, `rathipriya` (cabang "Our Preprocessing") memakai argmax ACF, dan
keduanya menghitung statistiknya pada deret penuh termasuk test.

Tabel di bawah menunjukkan bahwa *k* berubah drastis hanya karena pilihan tersebut.
Ini bukan detail teknis: bila *k* tidak stabil, kontribusi tahap rekayasa fitur pada
metode usulan juga tidak stabil, dan itu harus dilaporkan.

In [ ]:
lag_table = []
for category in CATEGORIES:
    y = data[category].to_numpy(dtype=float)
    row = {"Category": category}
    for rule in ["pacf_train", "pacf_full", "acf_train", "acf_full", "pacf_significant"]:
        row[rule] = P.select_lag(y, rule=rule)
    lag_table.append(row)

lag_table = pd.DataFrame(lag_table)
print("k (jumlah lag) menurut aturan seleksi -- *_full memakai data test (bocor)")
lag_table

## 2. Desain split

Split dicetak secara eksplisit (tanggal awal/akhir setiap blok, ukuran setiap blok)
karena reviewer secara khusus meminta tanggal train/test yang hilang dari naskah.
Perhatikan bahwa `A_lag1` dan `B_rich` berbagi **baris dan tanggal yang persis sama**:
kedua set fitur dibangun dari kerangka yang sama dengan *k* yang sama, sehingga
satu-satunya yang berbeda adalah kolom fiturnya.

In [ ]:
datasets = {}
split_rows = []
for category in CATEGORIES:
    for feature_set in P.FEATURE_SETS:
        d = P.build_pharma_dataset(data, category, feature_set,
                                   seasonal_period=SEASONAL_PERIOD,
                                   lag_rule="pacf_train")
        datasets[(category, feature_set)] = d
        split_rows.append({**d.describe(), "features": ", ".join(d.feature_names[:4]) +
                           (" ..." if len(d.feature_names) > 4 else "")})

split_table = pd.DataFrame(split_rows)
split_table.to_csv(f"../results/{EXPERIMENT}_splits.csv", index=False)
split_table

## 3. Menjalankan seluruh model

Setiap model dieksekusi lewat `P.run_model`, yang menegakkan kontrak tuning:
grid dievaluasi pada validation, konfigurasi terbaik di-refit pada train+val, lalu
test diprediksi tepat satu kali. Tidak ada jalur kode di mana test dapat memengaruhi
pemilihan hyperparameter.

`XGBoost`, `LR+XGB (rata-rata)` dan `LR-XGB (residual)` memakai grid yang sama
(`GRID_XGB_PHARMA`, 12 konfigurasi) agar anggaran tuningnya setara.

In [ ]:
MODELS = [
    # (nama, fungsi fit_predict, grid, jenis scaler)
    ("LR",                  P.fp_linear_regression, None,               None),
    ("GRNN",                P.fp_grnn,              P.GRID_GRNN,        "standard"),
    ("P_NN",                P.fp_pnn,               P.GRID_PNN,         "standard"),
    ("RBFNN",               P.fp_rbfnn,             P.GRID_RBFNN,       "standard"),
    ("XGBoost",             P.fp_xgboost,           P.GRID_XGB_PHARMA,  None),
    ("LR+XGB (average)",    P.fp_lr_xgb_average,    P.GRID_XGB_PHARMA,  None),
    ("LR-XGB (residual)",   P.fp_lr_xgb_residual,   P.GRID_XGB_PHARMA,  None),
]

rows = []
for category in CATEGORIES:
    for feature_set in P.FEATURE_SETS:
        d = datasets[(category, feature_set)]
        rows += P.naive_rows(d)
        for name, fn, grid, scaler in MODELS:
            rows.append(P.run_model(name, fn, d, grid, scaler_kind=scaler))
    print(f"selesai: {category}", flush=True)

# ARIMA(5,1,0) bersifat univariat: tidak bergantung pada set fitur, dijalankan sekali
# per kategori dan dilaporkan pada kedua set fitur dengan penanda eksplisit.
for category in CATEGORIES:
    d = datasets[(category, P.FEATURE_SET_A)]
    try:
        rows.append(P.run_model("ARIMA(5,1,0)", P.fp_arima, d,
                                {"order": [(5, 1, 0)]},
                                extra={"note": "univariat; tidak memakai fitur"}))
    except Exception as exc:
        print(f"ARIMA gagal untuk {category}: {exc}")

results = P.save_results(rows, EXPERIMENT)
print(f"{len(results)} baris hasil ditulis ke ../results/{EXPERIMENT}.csv")
results.head()

## 4. Tabel utama - RMSE test per kategori x set fitur

Ini adalah tabel yang menggantikan Tabel 2 pada naskah. Semua sel dihasilkan dari
satu proses, satu split, satu protokol tuning, dan satu seed.

In [ ]:
pivot = results.pivot_table(index=["category", "feature_set"], columns="model",
                            values="test_RMSE")
order = ["Naive", f"SeasonalNaive(s={SEASONAL_PERIOD})", "ARIMA(5,1,0)", "LR",
         "GRNN", "P_NN", "RBFNN", "XGBoost", "LR+XGB (average)", "LR-XGB (residual)"]
pivot = pivot[[c for c in order if c in pivot.columns]]
display(pivot.round(4))

winner = pivot.idxmin(axis=1).rename("model terbaik (RMSE test)")
print("\nModel terbaik per (kategori, set fitur):")
print(winner.to_string())
print("\nBerapa kali metode usulan menang:",
      int((winner == "LR-XGB (residual)").sum()), "dari", len(winner))

## 5. Apakah selisihnya signifikan? (Diebold-Mariano)

Selisih RMSE beberapa persen pada n_test kecil tidak otomatis berarti model lebih baik.
Uji DM di bawah membandingkan metode usulan dengan (a) baseline terbaik non-hibrida
pada kondisi yang sama dan (b) Seasonal Naive. Nilai DM negatif berarti metode usulan
lebih akurat; `p_value` di atas 0.05 berarti selisihnya tidak dapat dibedakan dari nol.

In [ ]:
by_key = {(r["category"], r["feature_set"], r["model"]): r for r in rows}
dm_rows = []
for category in CATEGORIES:
    for feature_set in P.FEATURE_SETS:
        prop = by_key.get((category, feature_set, "LR-XGB (residual)"))
        if prop is None:
            continue
        d = datasets[(category, feature_set)]
        competitors = [m for m in ["LR", "GRNN", "P_NN", "RBFNN", "XGBoost",
                                   f"SeasonalNaive(s={SEASONAL_PERIOD})"]
                       if (category, feature_set, m) in by_key]
        best = min(competitors,
                   key=lambda m: by_key[(category, feature_set, m)]["test_RMSE"])
        for other in {best, f"SeasonalNaive(s={SEASONAL_PERIOD})"}:
            ref = by_key[(category, feature_set, other)]
            test = P.diebold_mariano(d.y_test, prop["_test_pred"], ref["_test_pred"])
            dm_rows.append({
                "category": category, "feature_set": feature_set,
                "pembanding": other,
                "RMSE usulan": round(prop["test_RMSE"], 4),
                "RMSE pembanding": round(ref["test_RMSE"], 4),
                "DM": round(test["DM"], 3) if test["DM"] == test["DM"] else None,
                "p_value": round(test["p_value"], 4) if test["p_value"] == test["p_value"] else None,
                "signifikan (a=0.05)": (test["p_value"] < 0.05) if test["p_value"] == test["p_value"] else None,
            })

dm_table = pd.DataFrame(dm_rows).sort_values(["category", "feature_set", "pembanding"])
dm_table.to_csv(f"../results/{EXPERIMENT}_dm_test.csv", index=False)
dm_table

## 6. Ablasi aturan seleksi lag

Faktor ketiga: apakah kesimpulan berubah bila *k* dipilih dengan aturan lain?
Bagian ini menjalankan ulang metode usulan dan LR di bawah lima aturan seleksi lag.
Bila peringkat model berubah-ubah antar aturan, klaim keunggulan harus dilemahkan
secara eksplisit di naskah.

In [ ]:
ablation_rows = []
for rule in ["pacf_train", "pacf_full", "acf_train", "acf_full", "pacf_significant"]:
    for category in CATEGORIES:
        d = P.build_pharma_dataset(data, category, P.FEATURE_SET_B,
                                   seasonal_period=SEASONAL_PERIOD, lag_rule=rule)
        ablation_rows.append(P.run_model("LR", P.fp_linear_regression, d))
        ablation_rows.append(P.run_model("LR-XGB (residual)", P.fp_lr_xgb_residual,
                                         d, P.GRID_XGB_PHARMA))
    print("selesai aturan:", rule, flush=True)

ablation = P.save_results(ablation_rows, f"{EXPERIMENT}_lag_ablation")
ablation.pivot_table(index=["category", "lag_rule"], columns="model",
                     values="test_RMSE").round(4)

## 7. Pemeriksaan determinisme

Klaim reproduktifitas harus diuji, bukan dinyatakan. Sel di bawah menjalankan ulang
metode usulan untuk seluruh kategori pada proses yang sama dan membandingkan prediksi
bit-per-bit dengan hasil pertama.

In [ ]:
P.set_global_seed()
identical = True
for category in CATEGORIES:
    d = datasets[(category, P.FEATURE_SET_B)]
    again = P.run_model("LR-XGB (residual)", P.fp_lr_xgb_residual, d, P.GRID_XGB_PHARMA)
    first = by_key[(category, P.FEATURE_SET_B, "LR-XGB (residual)")]
    same = np.array_equal(again["_test_pred"], first["_test_pred"])
    identical &= same
    print(f"{category:6s} prediksi identik: {same}  | RMSE {again['test_RMSE']:.6f} "
          f"vs {first['test_RMSE']:.6f}")

print("\nSEMUA IDENTIK:", identical)

## 8. Catatan untuk naskah

* Semua angka di notebook ini berada pada **skala asli** unit penjualan daily; tidak
  ada metrik yang dilaporkan pada skala transformasi tanpa penanda. Untuk setiap baris,
  `RMSE = sqrt(MSE)` secara eksak menurut konstruksi (`compute_metrics`), sehingga
  ketidakkonsistenan RMSE/MSE yang ditemukan reviewer tidak dapat terulang.
* `results/{EXPERIMENT}.csv` berisi satu baris per (kategori, set fitur, model) lengkap
  dengan hyperparameter terpilih, metrik validation, metrik test, ukuran split, tanggal
  split, dan seed - format machine-readable yang diminta reviewer.
* Bila kolom `n_features` untuk `A_lag1` dan `B_rich` bernilai sama pada suatu kategori,
  itu berarti argmax PACF pada blok training bernilai 1 sehingga `rolling_mean_1`
  identik dengan `lag_1` dan dibuang sebagai kolom duplikat. Kondisi ini dilaporkan
  apa adanya: untuk kategori tersebut pipeline "kaya" memang berdegenerasi menjadi
  pipeline referensi.